# Lab 20 — Multi-Agent Research Demo Notebook

Notebook này giúp bạn **thử nghiệm nhanh** các khối logic của bài lab trước khi implement chính thức trong `src/`.

**Luồng làm việc:**
1. Khám phá schemas & shared state
2. Mock services (LLM + Search) để chạy không cần API key
3. Viết các agent demo (Researcher → Analyst → Writer)
4. Supervisor routing + vòng lặp workflow mini
5. Benchmark single-agent vs multi-agent

> ⚠️ **Quy tắc:** Notebook chỉ để prototype. Sau khi chạy được ở đây, bạn phải **chuyển logic vào `src/multi_agent_research_lab/`** và pass tests. Các ô có `TODO(student)` là phần bạn phải tự viết.

## 0. Setup

Chạy từ repo root với package đã cài (`pip install -e ".[dev]"`).

In [1]:
import sys
from pathlib import Path

# Cho phép import package khi chạy notebook từ thư mục notebooks/
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))

from multi_agent_research_lab.core.errors import StudentTodoError
from multi_agent_research_lab.core.schemas import (
    AgentName,
    AgentResult,
    BenchmarkMetrics,
    ResearchQuery,
    SourceDocument,
)
from multi_agent_research_lab.core.state import ResearchState

print("✅ Import OK — package sẵn sàng")

✅ Import OK — package sẵn sàng


## 1. Khám phá Shared State

`ResearchState` là **single source of truth** được truyền qua mọi agent. Mỗi agent đọc state, cập nhật, rồi trả lại.

In [2]:
query = ResearchQuery(
    query="So sánh RAG và fine-tuning cho domain adaptation",
    max_sources=3,
)
state = ResearchState(request=query)

state.record_route("researcher")
state.add_trace_event("demo", {"note": "first route recorded"})

print("Iteration:", state.iteration)
print("Route history:", state.route_history)
print("Trace:", state.trace)

Iteration: 1
Route history: ['researcher']
Trace: [{'name': 'demo', 'payload': {'note': 'first route recorded'}}]


## 2. Mock Services

Để demo không cần API key, ta dùng mock. Trong bản chính thức (`src/services/`), bạn sẽ nối provider thật (OpenAI / Tavily...).

- `MockSearchClient`: **đã viết sẵn** làm mẫu.
- `MockLLMClient`: **đã hoàn thành** — phân nhánh theo vai trò (analyst / writer) trong system prompt, trả nội dung giả lập + ước lượng token.

In [3]:
from dataclasses import dataclass


class MockSearchClient:
    """Trả về nguồn giả lập — cùng interface với services.search_client.SearchClient."""

    _FAKE_DOCS = [
        SourceDocument(
            title="RAG vs Fine-tuning: A Practical Guide",
            url="https://example.com/rag-vs-ft",
            snippet="RAG phù hợp khi dữ liệu thay đổi thường xuyên; fine-tuning tốt cho style/format.",
        ),
        SourceDocument(
            title="Retrieval-Augmented Generation Survey",
            url="https://example.com/rag-survey",
            snippet="RAG giảm hallucination bằng cách grounding vào tài liệu ngoài.",
        ),
        SourceDocument(
            title="When to Fine-tune LLMs",
            url="https://example.com/when-finetune",
            snippet="Fine-tuning hiệu quả khi cần hành vi nhất quán và latency thấp.",
        ),
    ]

    def search(self, query: str, max_results: int = 5) -> list[SourceDocument]:
        return self._FAKE_DOCS[:max_results]


@dataclass(frozen=True)
class MockLLMResponse:
    content: str
    input_tokens: int | None = None
    output_tokens: int | None = None


class MockLLMClient:
    """Giả lập LLM — cùng interface với services.llm_client.LLMClient."""

    def complete(self, system_prompt: str, user_prompt: str) -> MockLLMResponse:
        # Phân nhánh theo vai trò trong system_prompt, giống cách các agent thật
        # nhận system prompt chuyên biệt trong src/agents/.
        role = system_prompt.lower()
        if "analyst" in role:
            content = (
                "Claims chính:\n"
                "1. RAG phù hợp khi dữ liệu thay đổi thường xuyên vì grounding vào "
                "tài liệu ngoài [1][2].\n"
                "2. Fine-tuning phù hợp khi cần hành vi/style nhất quán và latency "
                "thấp [3].\n"
                "3. Evidence yếu: chưa nguồn nào đưa số liệu định lượng so sánh chi "
                "phí giữa hai hướng."
            )
        elif "writer" in role:
            content = (
                "RAG và fine-tuning giải quyết hai vấn đề khác nhau của domain "
                "adaptation. RAG grounding câu trả lời vào tài liệu ngoài nên hợp "
                "với dữ liệu thay đổi thường xuyên và giảm hallucination [1][2]. "
                "Fine-tuning nhúng hành vi vào trọng số nên hợp khi cần style nhất "
                "quán và latency thấp [3]. Thực tế thường kết hợp cả hai: fine-tune "
                "cho format, RAG cho kiến thức mới."
            )
        else:
            content = f"Mock single-shot answer for: {user_prompt[:100]}"
        return MockLLMResponse(
            content=content,
            input_tokens=(len(system_prompt) + len(user_prompt)) // 4,
            output_tokens=len(content) // 4,
        )


# Smoke test phần đã cho sẵn
search_client = MockSearchClient()
docs = search_client.search(query.query, max_results=query.max_sources)
for d in docs:
    print(f"- {d.title}: {d.snippet[:60]}...")

- RAG vs Fine-tuning: A Practical Guide: RAG phù hợp khi dữ liệu thay đổi thường xuyên; fine-tuning t...
- Retrieval-Augmented Generation Survey: RAG giảm hallucination bằng cách grounding vào tài liệu ngoà...
- When to Fine-tune LLMs: Fine-tuning hiệu quả khi cần hành vi nhất quán và latency th...


## 3. Demo Agents

Mỗi agent tuân theo contract `BaseAgent.run(state) -> state`.

- `DemoResearcherAgent`: **đã viết sẵn** làm mẫu — gọi search, ghi `sources` + `research_notes`.
- `DemoAnalystAgent`: **đã hoàn thành** — tổng hợp `sources` thành `analysis_notes`, guard khi thiếu nguồn.
- `DemoWriterAgent`: **đã hoàn thành** — viết `final_answer` kèm citation `[n]` + mục Sources.

In [4]:
class DemoResearcherAgent:
    """MẪU: thu thập nguồn và ghi chú nghiên cứu."""

    name = "researcher"

    def __init__(self, search_client: MockSearchClient) -> None:
        self.search_client = search_client

    def run(self, state: ResearchState) -> ResearchState:
        docs = self.search_client.search(
            state.request.query, max_results=state.request.max_sources
        )
        state.sources = docs
        state.research_notes = "\n".join(f"- {d.title}: {d.snippet}" for d in docs)
        state.agent_results.append(
            AgentResult(
                agent=AgentName.RESEARCHER,
                content=state.research_notes,
                metadata={"num_sources": len(docs)},
            )
        )
        state.add_trace_event("researcher.done", {"num_sources": len(docs)})
        return state


class DemoAnalystAgent:
    """Phân tích sources thành analysis_notes (không viết final answer)."""

    name = "analyst"

    def __init__(self, llm_client: MockLLMClient) -> None:
        self.llm_client = llm_client

    def run(self, state: ResearchState) -> ResearchState:
        if not state.sources:
            state.errors.append("analyst: no sources to analyze")
            return state
        source_block = "\n".join(
            f"[{i}] {d.title}: {d.snippet}"
            for i, d in enumerate(state.sources, start=1)
        )
        response = self.llm_client.complete(
            system_prompt=(
                "You are an analyst. Extract the main claims from the sources, "
                "cross-check them, and flag weak evidence. "
                "Do NOT write the final answer."
            ),
            user_prompt=f"Question: {state.request.query}\n\nSources:\n{source_block}",
        )
        state.analysis_notes = response.content
        state.agent_results.append(
            AgentResult(
                agent=AgentName.ANALYST,
                content=response.content,
                metadata={"num_sources": len(state.sources)},
            )
        )
        state.add_trace_event("analyst.done", {"chars": len(response.content)})
        return state


class DemoWriterAgent:
    """Viết final_answer có trích dẫn nguồn [n] + mục Sources."""

    name = "writer"

    def __init__(self, llm_client: MockLLMClient) -> None:
        self.llm_client = llm_client

    def run(self, state: ResearchState) -> ResearchState:
        context = state.analysis_notes or state.research_notes or ""
        response = self.llm_client.complete(
            system_prompt=(
                "You are a technical writer. Write the final answer for the given "
                "audience, citing sources as [n]. Never invent sources."
            ),
            user_prompt=(
                f"Question: {state.request.query}\n"
                f"Audience: {state.request.audience}\n\nNotes:\n{context}"
            ),
        )
        citations = "\n".join(
            f"[{i}] {d.title} ({d.url})"
            for i, d in enumerate(state.sources, start=1)
        )
        state.final_answer = f"{response.content}\n\nSources:\n{citations}"
        state.agent_results.append(
            AgentResult(
                agent=AgentName.WRITER,
                content=state.final_answer,
                metadata={"num_citations": len(state.sources)},
            )
        )
        state.add_trace_event("writer.done", {"answer_chars": len(state.final_answer)})
        return state


# Smoke test agent mẫu
state = ResearchState(request=query)
state = DemoResearcherAgent(search_client).run(state)
print(state.research_notes)

- RAG vs Fine-tuning: A Practical Guide: RAG phù hợp khi dữ liệu thay đổi thường xuyên; fine-tuning tốt cho style/format.
- Retrieval-Augmented Generation Survey: RAG giảm hallucination bằng cách grounding vào tài liệu ngoài.
- When to Fine-tune LLMs: Fine-tuning hiệu quả khi cần hành vi nhất quán và latency thấp.


## 4. Supervisor Routing

Supervisor quyết định agent nào chạy tiếp dựa trên state hiện tại. Đây là **trái tim của bài lab** — bạn tự thiết kế policy.

In [5]:
MAX_ITERATIONS = 6


def demo_supervisor_route(state: ResearchState) -> str:
    """Trả về một trong: 'researcher' | 'analyst' | 'writer' | 'done'."""
    # Guard chống vòng lặp vô hạn — GIỮ NGUYÊN dòng này
    if state.iteration >= MAX_ITERATIONS:
        return "done"

    # Fallback: một worker đã fail → không retry mãi, đi thẳng tới writer
    # để trả kết quả với dữ liệu đang có (giống policy trong src/agents/supervisor.py).
    if state.errors:
        return "done" if state.final_answer else "writer"

    # Routing chính: field nào còn thiếu là tín hiệu cho bước tiếp theo.
    if not state.sources:
        return "researcher"
    if not state.analysis_notes:
        return "analyst"
    if not state.final_answer:
        return "writer"
    return "done"


# Smoke test: state mới phải route tới researcher
assert demo_supervisor_route(ResearchState(request=query)) == "researcher"
print("✅ demo_supervisor_route sẵn sàng")

✅ demo_supervisor_route sẵn sàng


## 5. Mini Workflow Loop

Vòng lặp điều phối **đã viết sẵn** — chỉ chạy được sau khi bạn hoàn thành các TODO ở trên. Đây chính là logic bạn sẽ chuyển thành LangGraph nodes/edges trong `graph/workflow.py`.

In [6]:
def run_demo_workflow(query_text: str) -> ResearchState:
    q = ResearchQuery(query=query_text, max_sources=3)
    state = ResearchState(request=q)

    llm = MockLLMClient()
    agents = {
        "researcher": DemoResearcherAgent(MockSearchClient()),
        "analyst": DemoAnalystAgent(llm),
        "writer": DemoWriterAgent(llm),
    }

    while True:
        route = demo_supervisor_route(state)
        state.record_route(route)
        if route == "done":
            break
        state = agents[route].run(state)

    return state


try:
    final_state = run_demo_workflow("So sánh RAG và fine-tuning cho domain adaptation")
    print("Route history:", final_state.route_history)
    print("\n=== FINAL ANSWER ===\n")
    print(final_state.final_answer)
except StudentTodoError as exc:
    print(f"⛔ Còn TODO chưa hoàn thành: {exc}")
    print("→ Quay lại các ô trên, implement xong rồi chạy lại ô này.")

Route history: ['researcher', 'analyst', 'writer', 'done']

=== FINAL ANSWER ===

RAG và fine-tuning giải quyết hai vấn đề khác nhau của domain adaptation. RAG grounding câu trả lời vào tài liệu ngoài nên hợp với dữ liệu thay đổi thường xuyên và giảm hallucination [1][2]. Fine-tuning nhúng hành vi vào trọng số nên hợp khi cần style nhất quán và latency thấp [3]. Thực tế thường kết hợp cả hai: fine-tune cho format, RAG cho kiến thức mới.

Sources:
[1] RAG vs Fine-tuning: A Practical Guide (https://example.com/rag-vs-ft)
[2] Retrieval-Augmented Generation Survey (https://example.com/rag-survey)
[3] When to Fine-tune LLMs (https://example.com/when-finetune)


## 6. Benchmark: Single-agent vs Multi-agent

Dùng `run_benchmark` từ package để so sánh. Baseline single-agent (1 lần gọi LLM, không search) **bạn tự viết**.

In [7]:
from multi_agent_research_lab.evaluation.benchmark import run_benchmark


def run_single_agent(query_text: str) -> ResearchState:
    """Baseline: một lần gọi LLM duy nhất, không search, không phân tích."""
    q = ResearchQuery(query=query_text, max_sources=3)
    state = ResearchState(request=q)
    response = MockLLMClient().complete(
        system_prompt="You are a helpful research assistant. Answer in a single shot.",
        user_prompt=query_text,
    )
    state.final_answer = response.content
    state.record_route("single")
    state.agent_results.append(
        AgentResult(agent=AgentName.SINGLE, content=response.content)
    )
    return state


def compute_citation_coverage(state: ResearchState | None) -> float:
    """Tỷ lệ nguồn trong state.sources được nhắc đến trong final_answer."""
    if state is None or not state.sources or not state.final_answer:
        return 0.0
    cited = sum(
        1
        for d in state.sources
        if d.title in state.final_answer
        or (d.url is not None and d.url in state.final_answer)
    )
    return cited / len(state.sources)


demo_query = "So sánh RAG và fine-tuning cho domain adaptation"

results: list[BenchmarkMetrics] = []
for run_name, runner in [
    ("single_agent", run_single_agent),
    ("multi_agent", run_demo_workflow),
]:
    st, metrics = run_benchmark(run_name, demo_query, runner)
    metrics.citation_coverage = compute_citation_coverage(st)
    results.append(metrics)

print(f"{'run':<15}{'latency (s)':<15}{'citation cov.':<15}")
for m in results:
    print(f"{m.run_name:<15}{m.latency_seconds:<15.3f}{m.citation_coverage!s:<15}")

run            latency (s)    citation cov.  
single_agent   0.000          0.0            
multi_agent    0.000          1.0            


## 7. Next Steps — chuyển sang `src/`

Khi notebook chạy end-to-end, chuyển logic vào code chính thức:

| Notebook | Đích trong `src/multi_agent_research_lab/` |
|---|---|
| `MockLLMClient` → provider thật | `services/llm_client.py` |
| `MockSearchClient` → provider thật | `services/search_client.py` |
| `DemoResearcherAgent` / `DemoAnalystAgent` / `DemoWriterAgent` | `agents/researcher.py`, `agents/analyst.py`, `agents/writer.py` |
| `demo_supervisor_route` | `agents/supervisor.py` |
| `run_demo_workflow` → LangGraph nodes/edges | `graph/workflow.py` |
| `compute_citation_coverage` + quality score | `evaluation/benchmark.py` |

Sau đó verify:
```bash
make lint && make test
python -m multi_agent_research_lab.cli run --query "..."
bash scripts/check_todos.sh   # đảm bảo không còn TODO trong src/
```